WARNING: Notebook was written during development. May not work as intended. Retained here only for completion and future reference.

In [ ]:
import dataclasses

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import crops.src.bc as _bc
import crops.src.contact as _contact
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.solver as _solver
import crops.src.utils as _utils

jax.config.update("jax_enable_x64", True)

### Geom

In [ ]:
rod_length = 1.0

num_elems = 40
num_nodes = num_elems + 1

fiber_offset = 0.1  # c [m]
fiber_radius = 0.02

offsets_local = np.array(
  [[fiber_offset, 0.0], [-fiber_offset, 0.0], [0.0, fiber_offset], [0.0, -fiber_offset]]
)
num_fibers = 4
area_per_fiber = np.pi * fiber_radius**2

rod = _mesher.build_rod_geometry(
  analytical_curve=lambda k: jnp.array([k * rod_length, 0.0, 0.0]),
  parametric_coords=jnp.linspace(0.0, 1.0, num_nodes),
  fiber_offsets=jnp.asarray(np.tile(offsets_local, (num_elems, 1, 1))),
  fiber_areas=jnp.full((num_elems, num_fibers), area_per_fiber),
  fiber_mass_density=jnp.ones((num_elems, num_fibers)),
  fiber_centroid_moi=jnp.zeros((num_elems, num_fibers, 2)),
  seed_binormal=jnp.array([0.0, 0.0, 1.0]),
)


In [ ]:
shear_modulus = 1.0e6  # G [Pa]

In [ ]:
youngs_modulus = 3.0 * shear_modulus
area_total = num_fibers * area_per_fiber
moi_11 = np.sum(area_per_fiber * offsets_local[:, 0] ** 2)
bending_stiffness = youngs_modulus * moi_11
shear_stiffness = shear_modulus * area_total
contact_radius = float(jnp.max(rod.contact_radii))

# The indenter presses on the LAST ELEMENT CENTRE, not the tip node, because wall
# contact is evaluated at element centres. Use that station for the analytical.
indent_station = float(rod.elem_centers[-1, 0])
compliance_at_station = (
  indent_station**3 / (3.0 * bending_stiffness) + indent_station / shear_stiffness
)
print(f"EI = {bending_stiffness:.3f},  GA = {shear_stiffness:.1f}")
print(f"contact radius = {contact_radius:.4f} m")
print(
  f"indent station = {indent_station:.4f} m, compliance = "
  f"{compliance_at_station:.6e} m/N"
)

### Circle indent

In [ ]:
def build_circle_sdf(
  circle_radius: float,
  half_extent: float,
  num_grid: int,
) -> tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
  """Sample the exact SDF of a circle centred at the body-frame origin.

              phi(x) = |x| - circle_radius
  phi > 0 outside the circle and phi < 0 inside

  Args:
    circle_radius: Radius of the indenter [m].
    half_extent: Half width of the square grid about the origin [m]. Must exceed
      `circle_radius + contact_radius`.
    num_grid: Number of grid nodes along each axis.

  Returns:
    A tuple `(sdf_grid, physical_origin, spacing)` of shapes `(num_grid, num_grid)`,
      `(num_dim=2,)` and `(num_dim=2,)`.
  """
  axis = jnp.linspace(-half_extent, half_extent, num_grid)
  mesh_x, mesh_y = jnp.meshgrid(axis, axis, indexing="ij")

  sdf_grid = jnp.sqrt(mesh_x**2 + mesh_y**2) - circle_radius
  spacing = jnp.full(2, axis[1] - axis[0])

  return sdf_grid, jnp.array([axis[0], axis[0]]), spacing

In [ ]:
circle_radius = 0.3
num_move_steps = 30

sdf_grid, physical_origin, spacing = build_circle_sdf(
  circle_radius=circle_radius,
  half_extent=circle_radius + 3.0 * contact_radius,
  num_grid=150,
)


barrier_distance = circle_radius + contact_radius
max_indentation = 0.15
start_height = barrier_distance + 0.05  # clear of the rod
end_height = barrier_distance - max_indentation  # pressed in

circle_heights = np.linspace(start_height, end_height, num_move_steps)
schedule = _contact.ObstacleMovement.from_translations(
  jnp.stack(
    [jnp.full(num_move_steps, indent_station), jnp.asarray(circle_heights)], axis=-1
  )
)

elem_length = rod_length / num_elems
wall_penalty_stiffness = 1.0e3 / (compliance_at_station * elem_length)

indenter = _contact.Obstacle(
  sdf_grid=sdf_grid,
  physical_origin=physical_origin,
  spacing=spacing,
  wall_penalty_stiffness=wall_penalty_stiffness,
  extrap_fill_value=10.0,
  movement=schedule,
)
print(f"barrier distance     = {barrier_distance:.4f} m")
print(f"circle centre height : {start_height:.4f} -> {end_height:.4f} m")
print(
  f"max indentation      = {max_indentation:.4f} m "
  f"({max_indentation / rod_length:.1%} of L)"
)
print(f"expected peak force  = {max_indentation / compliance_at_station:.3f} N")

### BC

In [ ]:
clamped_dofs = (
  _utils.Translation.X,
  _utils.Translation.Y,
  _utils.Translation.Z,
  _utils.Rotation.THETA_X,
  _utils.Rotation.THETA_Y,
  _utils.Rotation.THETA_Z,
)
clamp = [
  _bc.DirichletBC(
    values=(dof, [np.array([0.0]) for _ in range(num_move_steps)]),
    node_targets=[np.array([0]) for _ in range(num_move_steps)],
    name=f"clamp_{dof}",
  )
  for dof in clamped_dofs
]

planar = [
  _bc.DirichletBC(
    values=(dof, [np.zeros(num_nodes) for _ in range(num_move_steps)]),
    node_targets=[np.arange(num_nodes) for _ in range(num_move_steps)],
    name=f"planar_{dof}",
  )
  for dof in (_utils.Translation.Z, _utils.Rotation.THETA_Y)
]
boundary_conditions = _bc.process_boundary_conditions(rod, clamp + planar)


### Solver

In [ ]:
nr_solver = _solver.NewtonRaphson(
  solver_type=_solver.NonlinearSolvers.ARMIJO_BACKTRACKING_NEWTON_RAPHSON,
  nonlinear_settings={
    "max_iter": 200,
    "threshold": 1e-6,
    "abs_threshold": 1e-6,
    "lam_min": 0.01,
    "line_search_max_iter": 15,
    "line_search_shrink": 0.5,
    "line_search_alpha_min": 1e-6,
    "line_search_armijo_c": 1e-4,
  },
  linear_settings={
    "solver": _solver.LinearSolvers.SCIPY_SPARSE,
    "rtol": 1e-12,
  },
)
problem = _fe.CosseratProblem(
  rod=rod,
  bc=boundary_conditions,
  nr_solver=nr_solver,
  obstacle=indenter,
)

In [ ]:
shear_moduli = jnp.full((num_elems, num_fibers), shear_modulus)

u_star, u_history = problem.solve(u0=jnp.zeros(rod.num_dofs), shear_moduli=shear_moduli)
history = np.asarray(u_history).reshape(num_move_steps, num_nodes, -1)

print(f"final tip y = {history[-1, -1, 1]:.6f} m")

In [ ]:
def elem_overlap_at_step(step):
  """Per-element overlap depth eps = r_contact - phi at the deformed element centres."""
  nodal = history[step]
  deformed_nodes = np.asarray(rod.node_coords) + nodal[:, :3]
  centres = jnp.asarray(0.5 * (deformed_nodes[:-1, :2] + deformed_nodes[1:, :2]))
  obj_trans, obj_rot_mat = indenter.pose_at_step(step)
  phi = indenter.compute_sdf(centres, obj_trans, obj_rot_mat)
  return np.asarray(rod.contact_radii) - np.asarray(phi)


elem_x = np.asarray(rod.elem_centers)[:, 0]
fig, ax = plt.subplots(figsize=(7, 4.5))
for step in range(0, num_move_steps, max(1, num_move_steps // 6)):
  overlap = elem_overlap_at_step(step)
  ax.plot(elem_x, np.maximum(overlap, 0.0), lw=1.2,
          label=f"$h$ = {circle_heights[step]:.3f}")
ax.axvline(indent_station, color="k", ls=":", lw=1, label="indenter axis")
ax.set_xlabel("$x$ [m]")
ax.set_ylabel(r"overlap $\varepsilon$ [m]")
ax.legend(fontsize=7)
plt.show()

In [ ]:
theta = np.linspace(0.0, 2.0 * np.pi, 200)

fig, ax = plt.subplots(figsize=(9, 5))
colors = plt.cm.viridis(np.linspace(0.0, 0.9, 5))

for color, step in zip(colors, np.linspace(0, num_move_steps - 1, 5, dtype=int)):

  shape = np.asarray(rod.node_coords) + history[step, :, :3]

  ax.plot(shape[:, 0], shape[:, 1], lw=1.8, color=color)
  ax.plot(indent_station + circle_radius * np.cos(theta),
          circle_heights[step] + circle_radius * np.sin(theta),
          lw=1.0, color=color, alpha=0.6)

ax.set_aspect("equal")
ax.set_xlabel("$x$ [m]")
ax.set_ylabel("$y$ [m]")
plt.show()

In [ ]:
station_y = np.array([
  0.5 * (history[k, num_elems - 1, 1] + history[k, num_elems, 1])
  for k in range(num_move_steps)
])
barrier_y = circle_heights - barrier_distance
indentation = np.maximum(-barrier_y, 0.0)
reaction_exact = indentation / compliance_at_station
reaction_solved = np.maximum(-station_y, 0.0) / compliance_at_station

fig, (ax_y, ax_f) = plt.subplots(1, 2, figsize=(11, 4))
ax_y.plot(circle_heights, station_y, "o-", ms=4, label="CROPS")
ax_y.plot(circle_heights, np.minimum(barrier_y, 0.0), "k--", lw=1.5,
          label="rigid barrier")
ax_y.axvline(barrier_distance, color="g", ls=":", label="predicted onset")
ax_y.set_xlabel("circle centre height $h$ [m]")
ax_y.set_ylabel("element-centre $y$ [m]"); ax_y.legend()

ax_f.plot(circle_heights, reaction_exact, "k-", lw=2, label="rigid indenter")
ax_f.plot(circle_heights, reaction_solved, "o", ms=4, mfc="none", label="penalty")
ax_f.set_xlabel("circle centre height $h$ [m]")
ax_f.set_ylabel("reaction [N]"); ax_f.legend()
fig.tight_layout(); plt.show()
print(f"contact onset predicted at h = {barrier_distance:.4f} m")